In [ ]:
import os
import math
import random
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Reduces allocator fragmentation -- set before any CUDA allocation
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# -- FLAGS --
RESUME_TRAINING = True  # True  -> load latest/best checkpoint and continue
                         # False -> start from scratch
VRAM_FRACTION   = 1.0   # fraction of VRAM PyTorch may use (0.85 = 10.2 GB on 12 GB)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    torch.cuda.set_per_process_memory_fraction(VRAM_FRACTION, device=0)
    print(f'Using device: {device}  |  VRAM cap: {VRAM_FRACTION*100:.0f}%')
else:
    print('Using device: cpu')

DATA_ROOT  = os.path.join(os.getcwd(), 'data')
MODEL_ROOT = os.path.join(os.getcwd(), 'models')
os.makedirs(MODEL_ROOT, exist_ok=True)

In [ ]:
from helper_functions.utils import PersonReIDDataset

# Prefer the pre-cropped copy if preprocess_iust.py has been run — otherwise
# fall back to the raw full-frame dataset (much slower; YOLO runs per image).
_CROPS_ROOT = os.path.join(DATA_ROOT, 'IUSTPersonReID_crops')
_RAW_ROOT   = os.path.join(DATA_ROOT, 'IUSTPersonReID')
IUST_ROOT   = _CROPS_ROOT if os.path.isdir(_CROPS_ROOT) else _RAW_ROOT
USE_PRECROPPED = (IUST_ROOT == _CROPS_ROOT)
print(f'IUST root: {IUST_ROOT}  (pre-cropped: {USE_PRECROPPED})')

dataset_train = PersonReIDDataset(os.path.join(IUST_ROOT, 'bounding_box_train'), min_seq_len=2)
dataset_test  = PersonReIDDataset(os.path.join(IUST_ROOT, 'bounding_box_test'),  min_seq_len=1)

all_seqs_train = [s for cams in dataset_train.sequences_by_person.values()
                    for seqs in cams.values() for s in seqs]
all_seqs_test  = [s for cams in dataset_test.sequences_by_person.values()
                    for seqs in cams.values() for s in seqs]

print(f'Train identities: {len(dataset_train.sequences_by_person)}')
print(f'Train sequences:  {len(all_seqs_train)}')
print(f'Test  identities: {len(dataset_test.sequences_by_person)}')
print(f'Test  sequences:  {len(all_seqs_test)}')

In [ ]:
import numpy as np

lens = [len(s) for s in all_seqs_train]
print(f'Train seq lengths -- mean: {np.mean(lens):.1f}  median: {np.median(lens):.0f}  '
      f'p95: {np.percentile(lens,95):.0f}  p99: {np.percentile(lens,99):.0f}  max: {max(lens)}')

In [ ]:
import torchvision.transforms as T
from torch.utils.data import Sampler

# If using pre-cropped images, no YOLO step needed — skip DetectionCropTransform.
# Otherwise fall back to live YOLO cropping (slow: ~1 YOLO call per frame loaded).
if USE_PRECROPPED:
    _pre_crop = []
else:
    from ultralytics import YOLO
    from helper_functions.utils import DetectionCropTransform
    yolo = YOLO('yolov8n.pt')
    _pre_crop = [DetectionCropTransform(yolo, conf_thresh=0.4, padding=0.05)]

MAX_SEQ_LEN = 64
P, K = 12, 4  # 24 sequences/batch

train_transform = T.Compose([
    *_pre_crop,
    T.Resize((256, 128)),
    T.RandomHorizontalFlip(),
    T.Pad(10),
    T.RandomCrop((256, 128)),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3)),
])

test_transform = T.Compose([
    *_pre_crop,
    T.Resize((256, 128)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


class SequenceLabelDataset(Dataset):
    """Returns (frames, length, person_id)."""
    def __init__(self, sequences_by_person, transform, max_seq_len):
        self.transform          = transform
        self.max_seq_len        = max_seq_len
        self.samples            = []
        self.pid_to_indices     = {}
        self.pid_cam_to_indices = {}

        for pid, cam_data in sequences_by_person.items():
            for cam, seqs in cam_data.items():
                for seq in seqs:
                    i = len(self.samples)
                    self.samples.append((seq, pid))
                    self.pid_to_indices.setdefault(pid, []).append(i)
                    self.pid_cam_to_indices.setdefault(pid, {}).setdefault(cam, []).append(i)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        seq, pid = self.samples[idx]
        paths  = [p.image_path for p in seq[:self.max_seq_len]]
        frames = [self._load(p) for p in paths]
        return torch.stack(frames), len(frames), pid

    def _load(self, path):
        return self.transform(Image.open(path).convert('RGB'))


class PKSampler(Sampler):
    """Camera-aware PK sampler -- prefers cross-camera sequences per person."""
    def __init__(self, pid_to_indices, pid_cam_to_indices, P=12, K=4):
        self.pid_to_indices     = pid_to_indices
        self.pid_cam_to_indices = pid_cam_to_indices
        self.P = P
        self.K = K
        self.pids = [pid for pid, idxs in pid_to_indices.items() if len(idxs) >= 2]

    def _sample_cross_camera(self, pid):
        cam_data = self.pid_cam_to_indices[pid]
        cameras  = list(cam_data.keys())
        if len(cameras) == 1:
            return random.choices(self.pid_to_indices[pid], k=self.K)
        random.shuffle(cameras)
        cam_cycle = cameras * (self.K // len(cameras) + 1)
        return [random.choice(cam_data[cam]) for cam in cam_cycle[:self.K]]

    def __iter__(self):
        pids = self.pids.copy()
        random.shuffle(pids)
        for i in range(0, len(pids) - self.P + 1, self.P):
            batch = []
            for pid in pids[i : i + self.P]:
                batch.extend(self._sample_cross_camera(pid))
            yield batch

    def __len__(self):
        return len(self.pids) // self.P


def sequence_label_collate(batch):
    imgs_list, lengths, pids = zip(*batch)
    max_t = max(lengths)
    C, H, W = imgs_list[0].shape[1:]
    padded = torch.zeros(len(imgs_list), max_t, C, H, W)
    for i, (imgs, t) in enumerate(zip(imgs_list, lengths)):
        padded[i, :t] = imgs
    return padded, torch.tensor(lengths, dtype=torch.long), torch.tensor(pids, dtype=torch.long)

In [ ]:
train_dataset = SequenceLabelDataset(
    dataset_train.sequences_by_person,
    transform=train_transform,
    max_seq_len=MAX_SEQ_LEN
)

sampler = PKSampler(train_dataset.pid_to_indices,
                    train_dataset.pid_cam_to_indices, P=P, K=K)

print(f'Train sequences:       {len(train_dataset)}')
print(f'Persons with >=2 seqs: {len(sampler.pids)}')
print(f'Steps per epoch:       {len(sampler)}')

train_loader = DataLoader(
    train_dataset,
    batch_sampler=sampler,
    collate_fn=sequence_label_collate,
    pin_memory=True,
    num_workers=0
)

In [ ]:
from helper_functions.model import ImprovedSeqToSeqReIDModel, CombinedReIDLoss

all_train_pids = sorted(dataset_train.sequences_by_person.keys())
pid_to_cls     = {pid: i for i, pid in enumerate(all_train_pids)}
NUM_CLASSES    = len(all_train_pids)
print(f'Training identities: {NUM_CLASSES}')

model = ImprovedSeqToSeqReIDModel(
    embedding_dim=512,
    rnn_hidden=512,
    num_classes=NUM_CLASSES
).to(device)

# Layer-wise LR decay across all 12 ViT blocks
vit_base_lr = 5e-6
decay       = 0.75

param_groups = []
param_groups.append({
    'params': list(model.encoder.vit.patch_embed.parameters()) +
              [model.encoder.vit.pos_embed, model.encoder.vit.cls_token],
    'lr': vit_base_lr * (decay ** 12)
})
for i, block in enumerate(model.encoder.vit.blocks):
    param_groups.append({'params': list(block.parameters()), 'lr': vit_base_lr * (decay ** (11 - i))})
param_groups.append({'params': list(model.encoder.vit.norm.parameters()), 'lr': vit_base_lr})

head_params = (
    list(model.encoder.global_proj.parameters()) +
    list(model.encoder.local_proj_2.parameters()) +
    list(model.encoder.local_proj_4.parameters()) +
    list(model.encoder.local_proj_8.parameters()) +
    list(model.encoder.temporal_tf.parameters()) +
    list(model.encoder.rnn.parameters()) +
    list(model.encoder.attention.parameters()) +
    list(model.encoder.embed_head.parameters()) +
    (list(model.classifier.parameters()) if model.classifier is not None else [])
)
param_groups.append({'params': head_params, 'lr': 5e-5})

WARMUP_EPOCHS = 5
optimizer = torch.optim.AdamW(param_groups, weight_decay=1e-4)
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=1e-3, end_factor=1.0, total_iters=WARMUP_EPOCHS
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-8
)
scaler    = torch.amp.GradScaler('cuda')
criterion = CombinedReIDLoss(margin=0.3, circle_m=0.25, circle_gamma=80,
                              lambda_circle=0.2, lambda_stripe=0.3,
                              am_scale=30.0, am_margin=0.35).to(device)

BEST_CKPT   = os.path.join(MODEL_ROOT, 'best_model_iust.pth')
LATEST_CKPT = os.path.join(MODEL_ROOT, 'latest_model_iust.pth')

# Optional warm start from MARS-trained backbone. Classifier shape differs
# (MARS 625 vs IUST NUM_CLASSES) so drop any classifier.* keys before loading;
# strict=False then lets the fresh IUST classifier stay randomly initialised.
MARS_PRETRAINED = os.path.join(MODEL_ROOT, 'best_model_mars.pth')

if RESUME_TRAINING:
    for ckpt in [LATEST_CKPT, BEST_CKPT]:
        if os.path.exists(ckpt):
            model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
            print(f'Resumed from {os.path.basename(ckpt)}')
            break
    else:
        if os.path.exists(MARS_PRETRAINED):
            state = torch.load(MARS_PRETRAINED, map_location=device, weights_only=True)
            state = {k: v for k, v in state.items() if not k.startswith('classifier.')}
            missing, unexpected = model.load_state_dict(state, strict=False)
            print(f'Warm-started encoder from {os.path.basename(MARS_PRETRAINED)}  '
                  f'(dropped classifier.*; {len(missing)} missing / {len(unexpected)} unexpected keys)')
        else:
            print('WARNING: RESUME_TRAINING=True but no checkpoint found. Starting fresh.')
else:
    print('Starting from scratch.')


In [ ]:
def train_epoch(model, loader, optimizer, scaler, device, pid_to_cls, criterion):
    model.train()
    total_loss = 0.0
    comp_sums  = {"id_loss": 0.0, "triplet_loss": 0.0,
                  "circle_loss": 0.0, "stripe_loss": 0.0}
    n = 0

    for imgs, lengths, pids in tqdm(loader, desc="Train"):
        imgs = imgs.to(device)
        pids = pids.to(device)
        cls_labels = torch.tensor([pid_to_cls[p.item()] for p in pids],
                                   dtype=torch.long, device=device)
        optimizer.zero_grad()
        with torch.amp.autocast("cuda"):
            embeddings, logits, stripe_feats = model(imgs, lengths)
            loss, loss_dict = criterion(embeddings, cls_labels, logits,
                                        stripe_feats=stripe_feats, lengths=lengths)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        for k in comp_sums:
            v = loss_dict[k]
            comp_sums[k] += v.item() if hasattr(v, "item") else float(v)
        n += 1

    avg_total = total_loss / n if n > 0 else 0.0
    avg_comps = {k: v / n for k, v in comp_sums.items()} if n > 0 else comp_sums
    return avg_total, avg_comps

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity as cos_sim
import gc

def encode_sequences(sequences, model, transform, device,
                      max_seq_len=None, batch_size=32, flip_tta=False):
    """Returns (embs, pids, camids). camids are needed so evaluate() can
    exclude same-(pid,camid) gallery matches per query (standard Market-1501
    protocol). Without this, near-duplicate same-camera frames leak and
    massively inflate Rank-1."""
    model.eval()

    def seq_len(seq):
        s = seq if max_seq_len is None else seq[:max_seq_len]
        return len(s)

    sequences = sorted(sequences, key=seq_len)
    all_embs, all_pids, all_camids = [], [], []

    def load_seq(seq, flipped=False):
        frames_to_use = seq if max_seq_len is None else seq[:max_seq_len]
        frames = []
        for p in frames_to_use:
            img = Image.open(p.image_path).convert('RGB')
            if flipped:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            frames.append(transform(img))
        return torch.stack(frames), len(frames), seq[0].person_id, seq[0].camera_id

    def run_batch(batch_seqs, flipped=False):
        results = [load_seq(s, flipped) for s in batch_seqs]
        imgs_list, lengths, pids, camids = zip(*results)
        max_t = max(lengths)
        C, H, W = imgs_list[0].shape[1:]
        padded = torch.zeros(len(imgs_list), max_t, C, H, W)
        for i, (imgs, t) in enumerate(zip(imgs_list, lengths)):
            padded[i, :t] = imgs
        lengths_t = torch.tensor(lengths, dtype=torch.long)
        with torch.amp.autocast('cuda'):
            embs = model(padded.to(device), lengths_t)
        return embs.cpu().float(), pids, camids

    with torch.no_grad():
        for start in tqdm(range(0, len(sequences), batch_size), desc='Encoding'):
            batch_seqs = sequences[start : start + batch_size]
            embs, pids, camids = run_batch(batch_seqs)
            if flip_tta:
                embs_flip, _, _ = run_batch(batch_seqs, flipped=True)
                import torch.nn.functional as F
                embs = F.normalize((embs + embs_flip) / 2, dim=1)
            all_embs.append(embs.numpy())
            all_pids.extend(pids)
            all_camids.extend(camids)

    return (np.concatenate(all_embs, axis=0),
            np.array(all_pids),
            np.array(all_camids))


def evaluate(q_embs, q_pids, q_camids, g_embs, g_pids, g_camids):
    """Cross-camera ReID eval: per query, exclude gallery items sharing the
    same (pid, camid). Standard Market-1501 protocol."""
    sim = cos_sim(q_embs, g_embs)
    rank1_hits, aps = [], []
    for i in range(len(q_pids)):
        keep = ~((g_pids == q_pids[i]) & (g_camids == q_camids[i]))
        if keep.sum() == 0:
            continue
        sim_i    = sim[i][keep]
        g_pids_i = g_pids[keep]
        order    = np.argsort(-sim_i)
        ranked   = g_pids_i[order]
        rank1_hits.append(ranked[0] == q_pids[i])
        rel = ranked == q_pids[i]
        if rel.sum():
            cum  = np.cumsum(rel)
            prec = cum / np.arange(1, len(rel) + 1)
            aps.append(prec[rel].mean())
    rank1 = float(np.mean(rank1_hits)) if rank1_hits else 0.0
    mAP   = float(np.mean(aps))        if aps        else 0.0
    return rank1, mAP

In [ ]:
# IUST ships an official query/ folder (Market-1501-style naming).
# Use it directly; bounding_box_test/ is the gallery.
query_base = PersonReIDDataset(os.path.join(IUST_ROOT, 'query'), min_seq_len=1)

query_seqs = [s for cams in query_base.sequences_by_person.values()
                for seqs in cams.values() for s in seqs]
gallery_seqs = all_seqs_test

print(f'Query sequences:   {len(query_seqs)}')
print(f'Gallery sequences: {len(gallery_seqs)}')

In [ ]:
NUM_EPOCHS = 100
PATIENCE   = 15

best_score       = 0.0   # rank1 + mAP — optimises both jointly
patience_counter = 0

for epoch in range(NUM_EPOCHS):
    train_loss, loss_comps = train_epoch(model, train_loader, optimizer, scaler,
                                         device, pid_to_cls, criterion)

    torch.save(model.state_dict(), LATEST_CKPT)

    gc.collect()
    torch.cuda.empty_cache()
    # Eval at short max_seq_len — YOLO-per-frame is the bottleneck.
    q_embs, q_pids, q_camids = encode_sequences(query_seqs,   model, test_transform, device,
                                                 max_seq_len=8, flip_tta=False)
    g_embs, g_pids, g_camids = encode_sequences(gallery_seqs, model, test_transform, device,
                                                 max_seq_len=8, flip_tta=False)
    model.train()
    gc.collect()
    torch.cuda.empty_cache()
    rank1, mAP = evaluate(q_embs, q_pids, q_camids, g_embs, g_pids, g_camids)
    score = rank1 + mAP

    print(f'Epoch [{epoch+1}/{NUM_EPOCHS}]  '
          f'loss={train_loss:.4f}  '
          f'id={loss_comps["id_loss"]:.4f}  '
          f'tri={loss_comps["triplet_loss"]:.4f}  '
          f'cir={loss_comps["circle_loss"] * criterion.lambda_circle:.4f}  '
          f'stripe={loss_comps["stripe_loss"] * criterion.lambda_stripe:.4f}  '
          f'Rank-1={rank1:.4f}  mAP={mAP:.4f}  score={score:.4f}')

    if epoch < WARMUP_EPOCHS:
        warmup_scheduler.step()
    else:
        scheduler.step(score)

    if score > best_score:
        best_score       = score
        patience_counter = 0
        torch.save(model.state_dict(), BEST_CKPT)
        print(f'  --> Saved best model  (score={best_score:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print('Early stopping.')
            break

print(f'Best score: {best_score:.4f}')

# --- Stripe health check (runs once after training ends) ---
print('\n=== Stripe similarity diagnostics ===')
model.eval()
with torch.no_grad():
    sample_imgs, sample_lengths, _ = next(iter(train_loader))
    sample_imgs = sample_imgs.to(device)
    import torch.nn.functional as F
    _, stripe_feats = model.encoder(sample_imgs, sample_lengths)
    sf = F.normalize(stripe_feats, dim=-1)  # (B, T, 4, D)

    for s in range(sf.shape[2]):
        sims = []
        for b in range(sf.shape[0]):
            L = sample_lengths[b].item()
            if L < 2:
                continue
            feat = sf[b, :L, s, :]
            sim  = feat @ feat.T
            off  = ~torch.eye(L, dtype=torch.bool, device=feat.device)
            sims.append(sim[off].mean().item())
        print(f'  Stripe {s} intra-sim (want > {criterion.stripe_loss.margin_intra}): {np.mean(sims):.3f}')

    B, T = sf.shape[0], sf.shape[1]
    sf_flat = sf.view(B * T, sf.shape[2], sf.shape[3])
    inter = torch.bmm(sf_flat, sf_flat.transpose(1, 2))  # (B*T, 4, 4)
    off_s = ~torch.eye(sf.shape[2], dtype=torch.bool, device=sf.device)
    print(f'  Inter-stripe sim  (want < {criterion.stripe_loss.margin:.1f}):  {inter[:, off_s].mean().item():.3f}')
model.train()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

model.load_state_dict(torch.load(BEST_CKPT, map_location=device, weights_only=True))


def re_ranking(q_embs, g_embs, k1=20, k2=6, lambda_value=0.3):
    feat = np.concatenate([q_embs, g_embs], axis=0)
    n_q, n = len(q_embs), len(q_embs) + len(g_embs)
    dist = 2 - 2 * (feat @ feat.T).clip(-1, 1)
    initial_rank = np.argsort(dist, axis=1)
    V = np.zeros_like(dist)
    for i in range(n):
        forward = set(initial_rank[i, 1:k1+1])
        R_i = [j for j in forward if i in set(initial_rank[j, 1:k1+1])]
        R_i_exp = list(R_i)
        for q in R_i:
            R_q = set(initial_rank[q, 1:int(k1/2)+1])
            if len(R_q) and len(R_q & set(R_i)) / len(R_q) >= 2/3:
                R_i_exp += list(R_q)
        R_i_exp = list(set(R_i_exp))
        w = np.exp(-dist[i, R_i_exp])
        w /= w.sum() + 1e-12
        V[i, R_i_exp] = w
    if k2 != 1:
        V_qe = np.zeros_like(V)
        for i in range(n):
            V_qe[i] = V[initial_rank[i, :k2]].mean(axis=0)
        V = V_qe
    jaccard = np.zeros((n_q, n - n_q))
    for i in range(n_q):
        for j in range(n_q, n):
            inter = np.minimum(V[i], V[j]).sum()
            union = np.maximum(V[i], V[j]).sum()
            jaccard[i, j - n_q] = 1 - inter / (union + 1e-12)
    return jaccard * (1 - lambda_value) + dist[:n_q, n_q:] * lambda_value


def evaluate_rerank(q_embs, q_pids, q_camids, g_embs, g_pids, g_camids,
                     k1=20, k2=6, lambda_value=0.3):
    """Same same-camera exclusion as evaluate(), applied after re-ranking."""
    dist = re_ranking(q_embs, g_embs, k1=k1, k2=k2, lambda_value=lambda_value)
    rank1_hits, aps = [], []
    for i in range(len(q_pids)):
        keep = ~((g_pids == q_pids[i]) & (g_camids == q_camids[i]))
        if keep.sum() == 0:
            continue
        d_i      = dist[i][keep]
        g_pids_i = g_pids[keep]
        order    = np.argsort(d_i)
        ranked   = g_pids_i[order]
        rank1_hits.append(ranked[0] == q_pids[i])
        rel = ranked == q_pids[i]
        if rel.sum():
            cum  = np.cumsum(rel)
            prec = cum / np.arange(1, len(rel) + 1)
            aps.append(prec[rel].mean())
    rank1 = float(np.mean(rank1_hits)) if rank1_hits else 0.0
    mAP   = float(np.mean(aps))        if aps        else 0.0
    return rank1, mAP


# Sweep eval-time max_seq_len to find the best length for this checkpoint.
# Checkpoint was selected during training at max_seq_len=8, so performance
# usually peaks near that range (out-of-distribution lengths hurt).
SEQ_LENS = [4, 8, 16, 32]
results  = {}

print('Eval-time max_seq_len sweep (query = gallery, flip TTA on, same-cam filtered)')
print('-' * 74)
print(f'{"seq_len":>7}  {"R-1":>7}  {"mAP":>7}  {"R-1(RR)":>8}  {"mAP(RR)":>8}  {"score":>7}')
print('-' * 74)

for L in SEQ_LENS:
    q_embs, q_pids, q_camids = encode_sequences(query_seqs,   model, test_transform, device,
                                                 max_seq_len=L, flip_tta=True)
    g_embs, g_pids, g_camids = encode_sequences(gallery_seqs, model, test_transform, device,
                                                 max_seq_len=L, flip_tta=True)
    r1, mAP = evaluate(q_embs, q_pids, q_camids, g_embs, g_pids, g_camids)
    r1_rr, mAP_rr = evaluate_rerank(q_embs, q_pids, q_camids,
                                     g_embs, g_pids, g_camids,
                                     k1=10, k2=2, lambda_value=0.25)
    results[L] = dict(q_embs=q_embs, q_pids=q_pids, q_camids=q_camids,
                      g_embs=g_embs, g_pids=g_pids, g_camids=g_camids,
                      rank1=r1, mAP=mAP, rank1_rr=r1_rr, mAP_rr=mAP_rr)
    print(f'{L:>7}  {r1:>7.4f}  {mAP:>7.4f}  {r1_rr:>8.4f}  {mAP_rr:>8.4f}  {r1+mAP:>7.4f}')

best_L = max(results, key=lambda L: results[L]['rank1'] + results[L]['mAP'])
print('-' * 74)
print(f'Best length: {best_L}  '
      f'(Rank-1={results[best_L]["rank1"]:.4f}  mAP={results[best_L]["mAP"]:.4f}  '
      f'RR: Rank-1={results[best_L]["rank1_rr"]:.4f}  mAP={results[best_L]["mAP_rr"]:.4f})')

# Use best length for CMC plot and downstream t-SNE in ma_0012.
q_embs   = results[best_L]['q_embs']
q_pids   = results[best_L]['q_pids']
q_camids = results[best_L]['q_camids']
g_embs   = results[best_L]['g_embs']
g_pids   = results[best_L]['g_pids']
g_camids = results[best_L]['g_camids']
rank1    = results[best_L]['rank1']
mAP      = results[best_L]['mAP']

# CMC with same-camera exclusion
sim      = cos_sim(q_embs, g_embs)
MAX_RANK = 20
cmc      = np.zeros(MAX_RANK)
n_valid  = 0
for i in range(len(q_pids)):
    keep = ~((g_pids == q_pids[i]) & (g_camids == q_camids[i]))
    if keep.sum() == 0:
        continue
    n_valid += 1
    order   = np.argsort(-sim[i][keep])
    matches = g_pids[keep][order] == q_pids[i]
    if not matches.any():
        continue
    for r in range(MAX_RANK):
        if matches[:r + 1].any():
            cmc[r:] += 1
            break
cmc /= max(n_valid, 1)

plt.figure(figsize=(8, 5))
plt.plot(range(1, MAX_RANK + 1), cmc, marker='o', linewidth=2, markersize=4)
plt.xlabel('Rank')
plt.ylabel('Identification Rate')
plt.title(f'CMC Curve -- IUSTPersonReID (max_seq_len={best_L}, cross-camera)')
plt.xticks(range(1, MAX_RANK + 1))
plt.ylim(0, 1)
plt.grid(True, alpha=0.4)
plt.tight_layout()
plt.savefig('iust_cmc.png', dpi=150)
plt.show()

print(f'Rank-1={cmc[0]:.4f}  Rank-5={cmc[4]:.4f}  Rank-10={cmc[9]:.4f}  Rank-20={cmc[19]:.4f}  mAP={mAP:.4f}')

In [ ]:
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import matplotlib.cm as cm

N_PERSONS     = 20
unique_pids   = np.unique(q_pids)
selected_pids = unique_pids[:N_PERSONS]

mask     = np.isin(q_pids, selected_pids)
embs_sub = q_embs[mask]
pids_sub = q_pids[mask]

tsne   = TSNE(n_components=2, perplexity=min(30, len(embs_sub) - 1),
              n_iter_without_progress=1000, random_state=42)
coords = tsne.fit_transform(embs_sub)

colors    = cm.tab20(np.linspace(0, 1, N_PERSONS))
pid_color = {pid: colors[i] for i, pid in enumerate(selected_pids)}

plt.figure(figsize=(10, 8))
for pid in selected_pids:
    idx = pids_sub == pid
    plt.scatter(coords[idx, 0], coords[idx, 1],
                color=pid_color[pid], label=f'ID {pid}', s=45, alpha=0.85)

plt.legend(loc='best', fontsize=6, ncol=2, framealpha=0.7)
plt.title('t-SNE -- Query Embeddings (IUSTPersonReID)')
plt.axis('off')
plt.tight_layout()
plt.savefig('iust_tsne.png', dpi=150)
plt.show()